In [1]:
# setup Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import multiprocessing

# Tự động đếm số lõi CPU thực tế trên máy bạn
cores = multiprocessing.cpu_count()
# Cấu hình số partition thường gấp 2-3 lần số lõi CPU để tối ưu hóa
safe_cores = max(4, int(cores * 0.6))
num_partitions = safe_cores * 3
# OPTIMIZED Spark Configuration
spark = SparkSession.builder \
    .appName("Amazon Review Local Processing") \
    .master(f"local[{safe_cores}]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "70MB") \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .getOrCreate()
print(f"✅ Spark {spark.version} initialized for LOCAL MODE")
print(f"🖥️  CPU Cores utilized: {cores}")
print(f"📊 Partitions configured: {num_partitions}")
print(f"💾 Driver Memory (Max RAM): 10GB")

✅ Spark 3.5.1 initialized for LOCAL MODE
🖥️  CPU Cores utilized: 16
📊 Partitions configured: 27
💾 Driver Memory (Max RAM): 10GB


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import DataFrame
import pandas as pd
import numpy as np
from pyspark.ml.feature import StringIndexer

In [3]:
from pathlib import Path

ROOT_DIR      = Path().resolve().parent
DATA_DIR      = ROOT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Đảm bảo thư mục processed tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 📥 Đường dẫn đầu vào (Ưu tiên dùng Parquet đã convert để nhanh hơn)
REVIEW_RAW_PARQUET = str(RAW_DIR / "Clothing_Shoes_and_Jewelry.parquet")
META_RAW_PARQUET   = str(RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.parquet")

# 📤 Đường dẫn đầu ra cho các bước xử lý tiếp theo
REVIEW_CLEAN_PARQUET = str(PROCESSED_DIR / "review_clean_spark.parquet")
META_CLEAN_PARQUET   = str(PROCESSED_DIR / "meta_clean_spark.parquet")
FINAL_REVIEWS_PARQUET = str(PROCESSED_DIR / "final_kcore_reviews.parquet")
FINAL_META_PARQUET = str(PROCESSED_DIR / "final_kcore_metadata.parquet")

# Các tham số cấu hình khác
K_CORE = 5
OUTLIER_ZSCORE_THRESHOLD = 3.0
MIN_TEXT_LENGTH = 10
MAX_TEXT_LENGTH = 5000
NUM_PARTITIONS = 200
print(f"📂 Project Root: {ROOT_DIR}")
print(f"✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.")

📂 Project Root: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis
✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.


In [4]:
import math
from pyspark.sql import functions as F

print("\n" + "="*60)
print("🚀 BẮT ĐẦU QUY TRÌNH XUẤT DỮ LIỆU CHO KAGGLE/COLAB")
print("="*60)
# ===========================================================
# 0. TỈA CỘT BẢNG REVIEWS (TỐI ƯU RAM)
# ===========================================================
# CHỈ XOÁ CÁC CỘT PHÁI SINH VÀ OUTLIER, GIỮ LẠI `timestamp` CHUẨN BỊ CHIA TẬP
cols_to_drop_reviews = [
    'review_date',
    'text_length', 'helpful_vote_outlier', 'text_length_outlier'
]
df_final_reviews = spark.read.parquet(FINAL_REVIEWS_PARQUET).drop(*cols_to_drop_reviews)
print(f"✅ Đã dọn dẹp các cột thừa khỏi bảng Reviews.")

# ===========================================================
# 1. TỈA CỘT BẢNG METADATA
# ===========================================================
df_final_meta = spark.read.parquet(FINAL_META_PARQUET)
cols_meta = [
    'parent_asin',    # Khóa (Bắt buộc để Join)
    'price',          # Biến số
    'average_rating', # Biến số
    'rating_number',  # Biến số
    'main_category',  # Biến phân loại
    'price_category', # Biến phân loại
    'store'           # Biến phân loại
]

# Tỉa cột và xóa duplicate
df_meta_final = df_final_meta.select(
    [c for c in cols_meta if c in df_final_meta.columns]
).dropDuplicates(['parent_asin'])
print(f"✅ Metadata columns kept: {df_meta_final.columns}")

# ===========================================================
# 2. MERGE DỮ LIỆU VÀ ĐÓNG BĂNG (CACHE)
# ===========================================================
print("\n🔗 Đang merge reviews + metadata và tải vào RAM...")

df_master = df_final_reviews.join(
    df_meta_final, 
    on='parent_asin', 
    how='inner'
)

# # BƯỚC CỨU MẠNG: Đóng băng dữ liệu sau khi xáo trộn để giữ nguyên thứ tự
# df_master = df_master.cache()

# # Kích hoạt tính toán và lưu vào RAM bằng 1 lệnh count duy nhất
# total_rows = df_master.count()
# print(f"Sau merge: {total_rows:,} reviews")


🚀 BẮT ĐẦU QUY TRÌNH XUẤT DỮ LIỆU CHO KAGGLE/COLAB
✅ Đã dọn dẹp các cột thừa khỏi bảng Reviews.
✅ Metadata columns kept: ['parent_asin', 'price', 'average_rating', 'rating_number', 'price_category', 'store']

🔗 Đang merge reviews + metadata và tải vào RAM...


In [5]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.window import Window
import pyspark.sql.functions as F

print("\n" + "="*60)
print("3. MÃ HÓA ID BẰNG TỪ ĐIỂN VÀ CHẶT ĐỨT DAG")
print("="*60)

# ===========================================================
# NGẮT DAG LẦN 1: Trích xuất Unique ID lưu xuống đĩa
# ===========================================================
print("🔄 Đang trích xuất Unique ID để tạo từ điển...")
df_master.select("user_id").distinct().coalesce(10).write.mode("overwrite").parquet(str(PROCESSED_DIR / "temp_unique_users.parquet"))
df_master.select("parent_asin").distinct().coalesce(10).write.mode("overwrite").parquet(str(PROCESSED_DIR / "temp_unique_items.parquet"))

unique_users = spark.read.parquet(str(PROCESSED_DIR / "temp_unique_users.parquet"))
unique_items = spark.read.parquet(str(PROCESSED_DIR / "temp_unique_items.parquet"))

# ===========================================================
# NGẮT DAG LẦN 2: Mã hóa và lưu file Mapping (Từ điển)
# ===========================================================
print("🔄 Đang học và tạo từ điển ID (Không xếp hạng để tối ưu tốc độ)...")
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_id_int", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="parent_asin", outputCol="item_id_int", handleInvalid="skip")

dict_users = user_indexer.fit(unique_users).transform(unique_users)
dict_items = item_indexer.fit(unique_items).transform(unique_items)

# Ép kiểu Integer
dict_users = dict_users.withColumn("user_id_int", F.col("user_id_int").cast("integer"))
dict_items = dict_items.withColumn("item_id_int", F.col("item_id_int").cast("integer"))

# Ghi thẳng từ điển xuống đĩa (Để dùng sau này đưa lên model)
dict_users.write.mode("overwrite").parquet(str(PROCESSED_DIR / "mapping_users.parquet"))
dict_items.write.mode("overwrite").parquet(str(PROCESSED_DIR / "mapping_items.parquet"))

final_user_map = spark.read.parquet(str(PROCESSED_DIR / "mapping_users.parquet"))
final_item_map = spark.read.parquet(str(PROCESSED_DIR / "mapping_items.parquet"))

# ===========================================================
# NGẮT DAG LẦN 3: JOIN VÀ LƯU MASTER XUỐNG ĐĨA TRƯỚC KHI SPLIT
# ===========================================================
print("🔗 Đang ghép ID chuẩn vào dữ liệu gốc...")
df_master = df_master.join(final_user_map, on="user_id", how="left")
df_master = df_master.join(final_item_map, on="parent_asin", how="left")

print("💾 Đang xuất Master Data đã mã hóa xuống đĩa (Chặt đứt DAG hoàn toàn)...")
df_master.write.mode("overwrite").parquet(str(PROCESSED_DIR / "temp_master_mapped.parquet"))

# Giải phóng triệt để rác trong RAM cũ
if df_master.is_cached:
    df_master.unpersist()




3. MÃ HÓA ID BẰNG TỪ ĐIỂN VÀ CHẶT ĐỨT DAG
🔄 Đang trích xuất Unique ID để tạo từ điển...
🔄 Đang học và tạo từ điển ID (Không xếp hạng để tối ưu tốc độ)...
🔗 Đang ghép ID chuẩn vào dữ liệu gốc...
💾 Đang xuất Master Data đã mã hóa xuống đĩa (Chặt đứt DAG hoàn toàn)...


In [6]:
print("\n" + "="*60)
print("4. CHIA TẬP DỮ LIỆU & XUẤT FINAL PARQUET")
print("="*60)

# Đọc lại từ file đã ngắt DAG -> Spark hiểu đây là một luồng data hoàn toàn mới và sạch
df_clean = spark.read.parquet(str(PROCESSED_DIR / "temp_master_mapped.parquet"))

# Cứu mạng Window Function: Chủ động ép Spark xáo trộn bộ nhớ theo user_id trước 
# Điều này tránh bị thắt cổ chai ở 1 CPU core khi chạy hàm Window
df_clean = df_clean.repartition(200, "user_id")

window_spec = Window.partitionBy("user_id").orderBy("timestamp")
df_clean = df_clean.withColumn("time_rank", F.percent_rank().over(window_spec))

TRAIN_PATH = str(PROCESSED_DIR / "train_data.parquet")
VAL_PATH   = str(PROCESSED_DIR / "val_data.parquet")
TEST_PATH  = str(PROCESSED_DIR / "test_data.parquet")

# Filter và ghi thẳng ra file (KHÔNG dùng cache hay count gì ở bước này)
print("⏳ Đang sắp xếp, cắt 70% và xuất TẬP TRAIN...")
df_clean.filter(F.col("time_rank") <= 0.70).drop("time_rank", "timestamp") \
        .write.mode('overwrite').parquet(TRAIN_PATH)

print("⏳ Đang sắp xếp, cắt 15% và xuất TẬP VAL...")
df_clean.filter((F.col("time_rank") > 0.70) & (F.col("time_rank") <= 0.85)).drop("time_rank", "timestamp") \
        .write.mode('overwrite').parquet(VAL_PATH)

print("⏳ Đang sắp xếp, cắt 15% và xuất TẬP TEST...")
df_clean.filter(F.col("time_rank") > 0.85).drop("time_rank", "timestamp") \
        .write.mode('overwrite').parquet(TEST_PATH)

print("\n✨ HOÀN THÀNH XUẤT DỮ LIỆU THÀNH CÔNG TỚI ĐÍCH!")


4. CHIA TẬP DỮ LIỆU & XUẤT FINAL PARQUET
⏳ Đang sắp xếp, cắt 70% và xuất TẬP TRAIN...
⏳ Đang sắp xếp, cắt 15% và xuất TẬP VAL...
⏳ Đang sắp xếp, cắt 15% và xuất TẬP TEST...

✨ HOÀN THÀNH XUẤT DỮ LIỆU THÀNH CÔNG TỚI ĐÍCH!


In [ ]:
import shutil
from pathlib import Path

# ... (code ghi file mapping của bạn ở trên) ...
dict_users.write.mode("overwrite").parquet(str(PROCESSED_DIR / "mapping_users.parquet"))
dict_items.write.mode("overwrite").parquet(str(PROCESSED_DIR / "mapping_items.parquet"))

print("✅ Đã tạo xong file Mapping chính thức!")

# Dọn rác (Xóa file temp gốc)
try:
    shutil.rmtree(str(PROCESSED_DIR / "temp_unique_users.parquet"))
    shutil.rmtree(str(PROCESSED_DIR / "temp_unique_items.parquet"))
    print("🗑️ Đã dọn dẹp các file Temp để giải phóng ổ cứng.")
except Exception as e:
    print(f"Lỗi khi xóa file: {e}")